# The core maths notebook
- Implementation of get_coords()
- Implementation of bivariate_k()

### Core concepts:
- K function includes parameters for:
    - Point pattern A (n_A points)
    - Point pattern B (n_B points)
    - Window area (|W|)
    - w_ij is an edge correction weight

Under CSR, K_AB (r) = πr^2

- The L-function is the vairance-stabilised transform
    L_AB(r) = sqrt(K_AB(r)/π) - r

Under CSR, L(r) = 0
Values above 0 indicate co-localisation at scale r

- Edge detection
    - Without it, points near the boundaries appear to have fewer neighbours than they really do
    - Simplest correction is to weight each pair by the inverse of the fraction of the circle centered at i with radius d_ij that falls inside W
        - e.g. if only 1/2 of the radius is inside W, multiply counts by 2

## Import dependencies and data

In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

fov3 = pd.read_parquet("../data/processed/fov3_strips.parquet")

# Gene pair: KRT8 x KRT18
# Chosen because they are obligate heterodimer partners in simple epithelial
# cells — strong prior expectation of co-localisation. This is our positive
# control: if the K-function doesn't detect co-localisation here, the
# implementation is wrong.
GENE_A = 'KRT8'
GENE_B = 'KRT18'

## The core functions

### get_coords()
- Retrieves coordinates of points for a given gene

In [2]:
def get_coords(df: pd.DataFrame, 
gene: str,
x_col: str = 'x_global_px',
y_col: str = 'y_global_px') -> np.ndarray:
    """Extract coordinates for a specific gene.
     Parameters:
     df : pd.DataFrame
         The dataframe containing the data. Must contain columns: x_col, y_col, and 'target'.
    gene : str
        Gene nams for coordinate extraction. Defaults match CosMx global coordiantes.
    

    Usage:
    get_coords(tissue_df, gene="KRT8")

    Returns:
    np.ndarray, shape (N, 2)
        An array of x and y coordinates for the specified gene. Returns (0,2) array if gene is absent.
     """
    coords = df.loc[df['target'] == gene, [x_col, y_col]].values
    return coords.astype(np.float64)

### get_window()
- Derives the bounding box window from a transcript dataframe


In [3]:
def get_window(df: pd.DataFrame,
                x_col: str = 'x_global_px',
                y_col: str = 'y_global_px') -> tuple:
    """
    Derive the counding box window from a transcript DataFrame.

    Usage:
    get_window(tissue_df)

    Returns:
    tuple: (x_min, x_max, y_min, y_max)
    """
    return (
        df[x_col].min(), df[x_col].max(),
        df[y_col].min(), df[y_col].max()
    )

### bivariate_k()
- Estimate the bivariate (cross-type) Ripley's K-function K_AB(r)

In [4]:
def bivariate_k(coords_a: np.ndarray,
                coords_b: np.ndarray,
                r_vals: np.ndarray,
                window: tuple) -> np.ndarray:
    """
    Bivariate (cross-type) Ripley's K-function K_AB(r).
    
    Estimates the expected number of B points within distance r of a 
    typical A point, normalised by the intensity of B. Under complete 
    spatial randomness (CSR / independent Poisson), K_AB(r) = pi * r^2.
    
    Uses Ripley's isotropic edge correction for rectangular windows:
    each (i, j) pair is weighted by 1 / (fraction of circumference of 
    circle centred at i with radius d_ij that lies inside the window).
    
    Parameters
    ----------
    coords_a : np.ndarray, shape (n_a, 2)
        Coordinates of pattern A (the "focal" pattern).
    coords_b : np.ndarray, shape (n_b, 2)
        Coordinates of pattern B (the "target" pattern).
    r_vals : np.ndarray, shape (n_r,)
        Radii at which to evaluate K.
    window : tuple
        (x_min, x_max, y_min, y_max) bounding rectangle.
    
    Returns
    -------
    np.ndarray, shape (n_r,)
        Estimated K_AB(r) at each radius.
    """
    x_min, x_max, y_min, y_max = window
    n_a = len(coords_a)
    n_b = len(coords_b)
    area = (x_max - x_min) * (y_max - y_min)
    lambda_b = n_b / area          # intensity of the target pattern
    
    # Build KD-tree on B for efficient range queries
    tree_b = cKDTree(coords_b)
    
    # For each A point, find all B neighbours within r_max
    r_max = r_vals[-1]
    # query_ball_point returns list of lists: indices of B points 
    # within r_max of each A point
    neighbours = tree_b.query_ball_point(coords_a, r_max)
    
    k_vals = np.zeros(len(r_vals))
    
    for idx_a in range(n_a):
        x_i, y_i = coords_a[idx_a]
        
        # Distances from this A point to each of its B neighbours
        b_indices = neighbours[idx_a]
        if len(b_indices) == 0:
            continue
        
        b_pts = coords_b[b_indices]
        dists = np.sqrt(((b_pts - coords_a[idx_a]) ** 2).sum(axis=1))
        
        # Edge correction: fraction of circumference inside window
        # for each neighbour distance
        dx_left  = x_i - x_min
        dx_right = x_max - x_i
        dy_bot   = y_i - y_min
        dy_top   = y_max - y_i
        
        for j, d_ij in enumerate(dists):
            if d_ij == 0:
                continue
            
            # Angle of arc OUTSIDE the window from each edge
            angle_outside = 0.0
            for dist_to_edge in [dx_left, dx_right, dy_bot, dy_top]:
                if dist_to_edge < d_ij:
                    # This edge clips the circle
                    angle_outside += 2.0 * np.arccos(
                        np.clip(dist_to_edge / d_ij, -1, 1)
                    )
            
            frac_inside = (2.0 * np.pi - angle_outside) / (2.0 * np.pi)
            frac_inside = max(frac_inside, 0.01)  # safety floor
            
            weight = 1.0 / frac_inside
            
            # This pair contributes to K(r) for all r >= d_ij
            for k, r in enumerate(r_vals):
                if d_ij <= r:
                    k_vals[k] += weight
    
    # Normalise
    k_vals /= (n_a * lambda_b)
    
    return k_vals

### k_to_l()
- Transforms K(r) to variance-stabilised L(r)

In [5]:
def k_to_l(k_vals: np.ndarray,
           r_vals: np.ndarray) -> np.ndarray:
    """            
    Transform K-function to variance-stabilised L-function.

    L(r) - sqrt(K(r) / π) - r

    Under CSR, L(r) = 0.
    L(r) > 0 : co-localisation at scale r.
    L(r) < 0 : spatial inhibition / repulsion at scale r.    
    """
    return np.sqrt(np.maximum(k_vals, 0) / np.pi) - r_vals

### compute_envelope()
- Monte Carlo permutation envelope for the bivariate L-function


In [6]:
def compute_envelope(coords_a: np.ndarray,
                     coords_b: np.ndarray,
                     r_vals: np.ndarray,
                     window: tuple,
                     n_sim: int = 99,
                     seed: int = 42) -> tuple:
    """
    Monte Carlo permutation envelope for the bivariate L-function.

    Null model: random permutation. Pool all coords_a and coords_b,
    randomly reassign n_a points to pattern A and n_b points to pattern B,
    recompute L(r). Repeat n_sim times to build a pointwise envelope.

    This null preserves total transcript intensity and spatial distrivbution
    while destroying any association *between* the two patterns.

    Parameters:
    n_sim : int
        99 simulations -> pointwise alhpa ~0.02 (observed exceeds all sims).
        Use 999 for publication figures.
    seed : int
        Fixed for reproducibility. Record in methods.

    Usage:
    l_lo, l_hi = compute_envelope(coords_a, coords_b, r_vals, window)

    Returns:
    l_lo, l_hi : np.ndarray, shape (n_r,)
        Pointwise min and max of simulation L(r) across all n_sim runs.
    """
    rng = np.random.default_rng(seed)
    n_a = len(coords_a)
    pooled = np.vstack([coords_a, coords_b])
    
    sim_l  = np.zeros((n_sim, len(r_vals)))

    for s in range(n_sim):
        idx = rng.permutation(len(pooled))
        sim_a = pooled[idx[:n_a]]
        sim_b = pooled[idx[n_a:]]
        k_sim = bivariate_k(sim_a, sim_b, r_vals, window)
        sim_l[s] = k_to_l(k_sim, r_vals)

    return sim_l.min(axis=0), sim_l.max(axis=0)

### plot_diagnostics()
- toggle via "DIAGNOSTICS = True/False"

In [12]:
def plot_diagnostics(coords_a: np.ndarray,
                     coords_b: np.ndarray,
                     window: tuple,
                     gene_a: str = 'Gene A',
                     gene_b: str = 'Gene B',
                     title: str = '',
                     ax=None) -> None:
    """
    Scatter plot of two transcript patterns with bounding-box window.

    Visual sanity check before running K-function analysis.
    Confirms: spatial distribution, window bounds, overlap/separation.

    Parameters
    ----------
    coords_a : np.ndarray, shape (n_a, 2)
        Coordinates of gene A transcripts.
    coords_b : np.ndarray, shape (n_b, 2)
        Coordinates of gene B transcripts.
    window : tuple
        (x_min, x_max, y_min, y_max) bounding box.
    gene_a, gene_b : str
        Gene names for the legend.
    title : str
        Plot title. If empty, auto-generated from gene names.
    ax : matplotlib Axes or None
        If None, creates a new figure. Pass an ax for multi-panel layouts.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 6))

    # --- Transcripts ---
    ax.scatter(coords_a[:, 0], coords_a[:, 1],
               s=8, alpha=0.6, color='steelblue',
               label=f'{gene_a} (n={len(coords_a)})')
    ax.scatter(coords_b[:, 0], coords_b[:, 1],
               s=8, alpha=0.6, color='tomato',
               label=f'{gene_b} (n={len(coords_b)})')

    # --- Bounding box ---
    x_min, x_max, y_min, y_max = window
    rect = plt.Rectangle((x_min, y_min),
                          x_max - x_min, y_max - y_min,
                          linewidth=1.5, edgecolor='black',
                          facecolor='none', linestyle='--',
                          label='Analysis window')
    ax.add_patch(rect)

    ax.set_xlabel('x_global_px')
    ax.set_ylabel('y_global_px')
    ax.set_title(title or f'{gene_a} × {gene_b}')
    ax.legend(fontsize=8, markerscale=2)
    ax.set_aspect('equal')

### Inital Test 0

In [7]:
coords_a = get_coords(fov3, gene=GENE_A)
coords_b = get_coords(fov3, gene=GENE_B)

window = get_window(fov3)

r_vals = np.linspace(0, 300, 60)
k_obs = bivariate_k(coords_a, coords_b, r_vals, window)
print(k_obs)
print(f"bivariate_k output shape: {k_obs.shape}")   # expect (60,)
print(f"K(r=0)   = {k_obs[0]:.2f}")                 # expect (0)

# Check against CSR theoretical at r=100 and r=200
for r_check in [100, 200]:
    idx = np.argmin(np.abs(r_vals - r_check))
    csr = np.pi * r_check**2
    print(f"K(r={r_check}) = {k_obs[idx]:.0f}  |  CSR = {csr:.0f}  |  ratio = {k_obs[idx]/csr:.2f}")
# ratio > 1 means more clustering than CSR — expected for KRT8/KRT18

l_obs = k_to_l(k_obs, r_vals)   
print(f"k_to_l output shape: {l_obs.shape}")   # expect (60,)
print(f"L(r=0)   = {l_obs[0]:.3f}")            # expect (0)

# Verify the transform manually at one point
idx_100 = np.argmin(np.abs(r_vals - 100))
manual  = np.sqrt(k_obs[idx_100] / np.pi) - 100
print(f"L(r=100) = {l_obs[idx_100]:.3f}  |  manual check = {manual:.3f}") # expect ~identical

# Synthetic CSR sanity check: if K = pi*r^2 exactly, L should = 0
k_csr   = np.pi * r_vals**2
l_csr   = k_to_l(k_csr, r_vals)
print(f"Max |L| for perfect CSR input: {np.abs(l_csr).max():.6f}") # expect ~0

l_lo, l_hi = compute_envelope(coords_a, coords_b, r_vals, window, 9)
print(f"Envelope shapes: l_lo={l_lo.shape}, l_hi={l_hi.shape}") # expect (60,) (60,)

# The envelope should straddle zero (simulations are under the null)
print(f"Envelope lo at r=100: {l_lo[np.argmin(np.abs(r_vals-100))]:.2f}")
print(f"Envelope hi at r=100: {l_hi[np.argmin(np.abs(r_vals-100))]:.2f}")
# Observed should sit above hi if there is genuine co-localisation
print(f"Observed  L at r=100: {l_obs[np.argmin(np.abs(r_vals-100))]:.2f}")

[     0.           1308.00621227   4708.82236418   9156.04348591
  18312.08697181  28514.53542753  36975.30551339  48224.15893893
  59377.39213931  74157.86233798  85537.51638475  98231.90583249
 110148.20586797 122749.10346653 134184.69152303 149006.67302698
 162609.93763461 174865.48140308 185373.61367123 200009.50123384
 215364.20223718 230358.69329909 243406.98077993 260347.86305106
 277490.24975631 292885.10408525 305060.1575666  319471.4878077
 337211.79071075 354770.28742253 370280.84626248 386036.98431549
 403031.30204302 423236.90791912 439062.78563215 455131.78564549
 467409.50721515 483835.3873139  499116.11504779 513221.17367225
 527966.49221325 542380.66816151 555104.73012668 567894.30150383
 581820.96864569 596748.9781732  611455.79211165 626066.32129522
 640704.96264178 655769.37948847 669278.89784652 679563.91266332
 692676.67629545 703486.26023645 716580.41088065 731949.62665487
 744254.065039   755939.88320781 769521.42085875 781017.50139864]
bivariate_k output shape:

### Inital test

In [8]:
STRIP = 'strip_2'
strip_df = fov3[fov3['strip'] == STRIP]

coords_a = get_coords(strip_df, GENE_A)
coords_b = get_coords(strip_df, GENE_B)
window   = get_window(strip_df)

print(f"{GENE_A}: {len(coords_a)} transcripts")
print(f"{GENE_B}: {len(coords_b)} transcripts")
print(f"Window: x=[{window[0]:.0f}, {window[1]:.0f}], "
      f"y=[{window[2]:.0f}, {window[3]:.0f}]")

narrow_dim = min(window[1]-window[0], window[3]-window[2])
print(f"Narrowest window dimension: {narrow_dim:.0f} px")
print(f"Recommended r_max: < {narrow_dim/2:.0f} px")

KRT8: 74 transcripts
KRT18: 118 transcripts
Window: x=[13052, 14252], y=[34219, 38450]
Narrowest window dimension: 1200 px
Recommended r_max: < 600 px


In [9]:
# 60 points gives smooth curve without excessive computation
r_vals = np.linspace(0, 300, 60)

k_obs = bivariate_k(coords_a, coords_b, r_vals, window)
l_obs = k_to_l(k_obs, r_vals)

# Spot-check before running the expensive envelope
print("L(r) at key scales:")
for r_check in [50, 100, 150, 200]:
    idx = np.argmin(np.abs(r_vals - r_check))
    print(f"  r={r_check}px: L={l_obs[idx]:.2f}  (CSR expectation: 0)")

L(r) at key scales:
  r=50px: L=130.66  (CSR expectation: 0)
  r=100px: L=181.09  (CSR expectation: 0)
  r=150px: L=211.37  (CSR expectation: 0)
  r=200px: L=224.84  (CSR expectation: 0)


### Diagnosing the inflated L(r) values

### Synthetic CSR test

In [11]:
def synthetic_csr_test(n_a=100, n_b=100, window=(0, 1000, 0, 1000), r_max=300, n_r=60):
    """Generate two independent uniform patterns and check L(r) ≈ 0."""
    rng = np.random.default_rng(42)
    x_min, x_max, y_min, y_max = window
    
    coords_a = np.column_stack([
        rng.uniform(x_min, x_max, n_a),
        rng.uniform(y_min, y_max, n_a)
    ])
    coords_b = np.column_stack([
        rng.uniform(x_min, x_max, n_b),
        rng.uniform(y_min, y_max, n_b)
    ])
    
    r_vals = np.linspace(0, r_max, n_r)
    k_obs = bivariate_k(coords_a, coords_b, r_vals, window)
    l_obs = k_to_l(k_obs, r_vals)
    
    print(f"Window: {window}")
    print(f"n_A={n_a}, n_B={n_b}")
    print(f"Max |L(r)|: {np.abs(l_obs).max():.2f}")
    print(f"L values at key scales:")
    for r_check in [50, 100, 200]:
        idx = np.argmin(np.abs(r_vals - r_check))
        print(f"  r={r_check}: L={l_obs[idx]:.2f}")
    
    return r_vals, l_obs

# Test 1: Square window (should work)
r1, l1 = synthetic_csr_test(window=(0, 1000, 0, 1000))

# Test 2: Elongated window matching strip geometry
r2, l2 = synthetic_csr_test(window=(0, 1200, 0, 4200))

Window: (0, 1000, 0, 1000)
n_A=100, n_B=100
Max |L(r)|: 12.05
L values at key scales:
  r=50: L=-1.53
  r=100: L=3.82
  r=200: L=2.48
Window: (0, 1200, 0, 4200)
n_A=100, n_B=100
Max |L(r)|: 9.44
L values at key scales:
  r=50: L=2.18
  r=100: L=-2.34
  r=200: L=4.61


Synthetic CSR validation: two independent Poisson patterns, L(r) ≈ 0 (max |L| < 13) for both square and elongated windows. Confirms unbiased K-function estimation